In [1]:
#IMPORTS

# If you need to install in a fresh kernel, uncomment:
# !pip install -q transformers datasets accelerate torch scikit-learn

import os
import csv
import json
import random
from dataclasses import dataclass

import numpy as np
import torch 
import torch.nn as nn

from datasets import load_dataset, DatasetDict, ClassLabel
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [3]:
#CONFIG

@dataclass
class Config:
    model_name: str = "roberta-large"    # or "roberta-base"
    max_length: int = 256
    batch_size: int = 32
    lr: float = 2e-5
    weight_decay: float = 0.0
    num_epochs: int = 20
    eval_interval_steps: int = 200           # evaluate every X optimizer steps
    patience: int = 5                        # early-stopping patience on eval acc
    warmup_ratio: float = 0.1
    save_dir: str = "./runs/ladder_cls"

cfg = Config()

os.makedirs(cfg.save_dir, exist_ok=True)
weights_path = os.path.join(cfg.save_dir, "weights.pt")
log_csv = os.path.join(cfg.save_dir, "training_logs.csv")
pred_json = os.path.join(cfg.save_dir, "predictions.json")

cfg

Config(model_name='roberta-large', max_length=256, batch_size=32, lr=2e-05, weight_decay=0.0, num_epochs=20, eval_interval_steps=200, patience=5, warmup_ratio=0.1, save_dir='./runs/ladder_cls')

In [4]:
import os

#output_path = "C:/Users/cyberai/analiseprojs/LadderDataSets/"
input_dir = "/home/jovyan/work/SentenceClassificationDataSets"



print("Files found:")
print(os.listdir(input_dir))

Files found:
['dev.csv', 'dev.jsonl', 'dev.txt', 'dev_t1055.csv', 'test.csv', 'test.jsonl', 'test.txt', 'train.csv', 'train.jsonl', 'train.txt', 'train_t1055_broad.csv']


In [7]:
#load dataset


# Set this to your uploads directory
input_dir = "/home/jovyan/work/SentenceClassificationDataSets/" 
data_files = {
    "train": os.path.join(input_dir, "train.txt"),
    "test": os.path.join(input_dir, "test.txt"),
    "validate": os.path.join(input_dir, "dev.txt"),
}

# Load the dataset; specify the separator
# Use sep="\t" for tab-separated, or sep="," for comma-separated
ladder_dataset = load_dataset("csv", data_files=data_files, sep="\t")

# Show structure
ladder_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2214
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 662
    })
    validate: Dataset({
        features: ['text', 'label'],
        num_rows: 568
    })
})

In [8]:
# Data Preview

TEXT_COL = "text"
LABEL_COL = "label"

for split_name, split in ladder_dataset.items():
    print(f"=== Split: {split_name} ===")
    print("Columns:", split.column_names)
    print("First 3 examples:")
    for i in range(min(3, len(split))):
        print(split[i])
    print()

=== Split: train ===
Columns: ['text', 'label']
First 3 examples:
{'text': 'Using the device accelerometer sensor it implements a simple pedometer that is used to measure movements of the victim.', 'label': 1}
{'text': 'When the malware is first started on the device it will begin by hiding its icon from the application drawer.', 'label': 1}
{'text': 'Then it will ask for the accessibility service privilege as visible in the following screenshot:', 'label': 1}

=== Split: test ===
Columns: ['text', 'label']
First 3 examples:
{'text': 'Activating the device’s microphone and listening to live conversations', 'label': 1}
{'text': 'Executing commands on the device', 'label': 1}
{'text': 'Copying files from the device to a Command & Control (C&C) center', 'label': 1}

=== Split: validate ===
Columns: ['text', 'label']
First 3 examples:
{'text': 'Since then, the implant’s functionality has been improving and remarkable new features implemented, such as the ability to record audio surrounding

In [9]:
def ensure_int_labels(ds, label_col):
    feat = ds["train"].features.get(label_col)
    if isinstance(feat, ClassLabel):
        id2label = {i: lab for i, lab in enumerate(feat.names)}
        label2id = {lab: i for i, lab in id2label.items()}
        return ds, label2id, id2label

    example = ds["train"][label_col][0]
    if isinstance(example, (int, np.integer)):
        unique = sorted(set(ds["train"][label_col]))
        id2label = {i: str(i) for i in unique}
        label2id = {str(i): i for i in unique}
        return ds, label2id, id2label

    # Map strings to ints
    all_labels = sorted(set(ds["train"][label_col]))
    id2label = {i: lab for i, lab in enumerate(all_labels)}
    label2id = {lab: i for i, lab in id2label.items()}

    def map_labels(example):
        example[label_col] = label2id[str(example[label_col])]
        return example

    ds = ds.map(map_labels)
    return ds, label2id, id2label

ladder_dataset, label2id, id2label = ensure_int_labels(ladder_dataset, LABEL_COL)
num_labels = len(id2label)
print("num_labels =", num_labels)
print("label2id (up to 10):", dict(list(label2id.items())[:10]))
print("id2label (up to 10):", dict(list(id2label.items())[:10]))

num_labels = 2
label2id (up to 10): {'0': 0, '1': 1}
id2label (up to 10): {0: '0', 1: '1'}


In [10]:
#Tokenizer and tokenization 

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)

def tokenize_fn(batch):
    return tokenizer(
        batch[TEXT_COL],
        padding=False,
        truncation=True,
        max_length=cfg.max_length,
    )

tokenized = ladder_dataset.map(tokenize_fn, batched=True)

# Rename label column to 'labels' as expected by HF models
if LABEL_COL != "labels":
    tokenized = tokenized.rename_column(LABEL_COL, "labels")

# Keep only needed columns
to_keep = ["input_ids", "attention_mask", "labels"]
existing_keep = [c for c in to_keep if c in tokenized["train"].column_names]
tokenized = tokenized.remove_columns([c for c in tokenized["train"].column_names if c not in existing_keep])

tokenized.set_format("torch")
tokenized

Map: 100%|██████████████████████████████████████████████████████████████████████████████████████████| 568/568 [00:00<00:00, 40988.33 examples/s]


DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2214
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 662
    })
    validate: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 568
    })
})

In [11]:
#Data Loader + Collator

from torch.utils.data import DataLoader

collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(tokenized["train"], batch_size=cfg.batch_size, shuffle=True, collate_fn=collator)
val_split_name = "validate" if "validate" in tokenized else "validation"
val_loader = DataLoader(tokenized[val_split_name], batch_size=cfg.batch_size, shuffle=False, collate_fn=collator)
test_loader = DataLoader(tokenized["test"], batch_size=cfg.batch_size, shuffle=False, collate_fn=collator)

len(train_loader), len(val_loader), len(test_loader)

(70, 18, 21)

In [12]:
# Loads pretrained classifier model head

model = AutoModelForSequenceClassification.from_pretrained(
    cfg.model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id={v: k for k, v in id2label.items()},
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

num_training_steps = cfg.num_epochs * len(train_loader)
num_warmup_steps = int(cfg.warmup_ratio * num_training_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

criterion = nn.CrossEntropyLoss()
criterion

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CrossEntropyLoss()

In [13]:
#EVALUATION

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    all_true, all_pred, all_text = [], [], []

    for batch in loader:
        labels = batch["labels"].to(device)
        batch_inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        outputs = model(**batch_inputs)
        loss = criterion(outputs.logits, labels)

        preds = outputs.logits.argmax(dim=-1)

        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_true.extend(labels.cpu().tolist())
        all_pred.extend(preds.cpu().tolist())

        try:
            decoded = tokenizer.batch_decode(batch_inputs["input_ids"], skip_special_tokens=True)
            all_text.extend(decoded)
        except Exception:
            all_text.extend([""] * labels.size(0))

    avg_loss = total_loss / total
    acc = total_correct / total

    # --- NEW: Precision, Recall, F1 (binary, pos_label=1 for "relevant") ---
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_true,
        all_pred,
        average="binary",   # binary task: relevant vs non-relevant
        pos_label=1,        # assumes 1 = relevant (positive class)
        zero_division=0
    )

    # return everything as before, plus P/R/F1 at the end
    return avg_loss, acc, np.array(all_true), np.array(all_pred), all_text, precision, recall, f1


In [14]:
#TRAINING LOOP  + EARLY STOPPING


best_val_acc = -1.0
no_improve = 0
global_step = 0

# Prepare CSV log
with open(log_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["mode", "epoch", "step", "loss", "acc"])

for epoch in range(cfg.num_epochs):
    model.train()
    running_loss, running_total, running_correct = 0.0, 0, 0

    print(f"--------------------------- Epoch {epoch} ---------------------------")
    for batch in train_loader:
        global_step += 1

        labels = batch["labels"].to(device)
        batch_inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        outputs = model(**batch_inputs)
        loss = criterion(outputs.logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        # Running stats
        bs = labels.size(0)
        running_loss += loss.item() * bs
        preds = outputs.logits.argmax(dim=-1)
        running_correct += (preds == labels).sum().item()
        running_total += bs

        # Periodic eval
        if global_step % cfg.eval_interval_steps == 0:
            train_loss = running_loss / running_total
            train_acc = running_correct / running_total
            print(f"Train @ step {global_step}: loss={train_loss:.4f}, acc={train_acc:.4f}")

            with open(log_csv, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["train", epoch, global_step, train_loss, train_acc])

            # Reset running stats
            running_loss, running_total, running_correct = 0.0, 0, 0

            
            val_loss, val_acc, _, _, _, val_P, val_R, val_F1 = evaluate(model, val_loader)
            print(f"Val @ step {global_step}: loss={val_loss:.4f}, acc={val_acc:.4f}")
            print(f"Val P={val_P:.4f}, R={val_R:.4f}, F1={val_F1:.4f}")

            with open(log_csv, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["val", epoch, global_step, val_loss, val_acc])

            # Early stopping check
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                no_improve = 0
                torch.save(model.state_dict(), weights_path)
            else:
                no_improve += 1
                if no_improve >= cfg.patience:
                    print("Early stopping triggered.")
                    break
    if no_improve >= cfg.patience:
        break

print("Training complete!")
print("Best Val Acc:", best_val_acc)

--------------------------- Epoch 0 ---------------------------
--------------------------- Epoch 1 ---------------------------
--------------------------- Epoch 2 ---------------------------
Train @ step 200: loss=0.2585, acc=0.8948
Val @ step 200: loss=0.3873, acc=0.8363
Val P=0.7994, R=0.8979, F1=0.8458
--------------------------- Epoch 3 ---------------------------
--------------------------- Epoch 4 ---------------------------
--------------------------- Epoch 5 ---------------------------
Train @ step 400: loss=0.0579, acc=0.9794
Val @ step 400: loss=0.5596, acc=0.8380
Val P=0.8636, R=0.8028, F1=0.8321
--------------------------- Epoch 6 ---------------------------
--------------------------- Epoch 7 ---------------------------
--------------------------- Epoch 8 ---------------------------
Train @ step 600: loss=0.0159, acc=0.9961
Val @ step 600: loss=0.9134, acc=0.8345
Val P=0.7879, R=0.9155, F1=0.8469
--------------------------- Epoch 9 ---------------------------
------------

In [15]:
# Reload best weights
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))

train_loss, train_acc, _, _, _, _, _, _ = evaluate(model, train_loader)
val_loss, val_acc, _, _, _, _, _, _ = evaluate(model, val_loader)
test_loss, test_acc, y_true, y_pred, test_text, _, _, _ = evaluate(model, test_loader)

print(f"Train: loss={train_loss:.4f}, acc={train_acc:.4f}")
print(f"Val:   loss={val_loss:.4f}, acc={val_acc:.4f}")
print(f"Test:  loss={test_loss:.4f}, acc={test_acc:.4f}")

# For PR/RE/F1: 'binary' for 2 classes, else 'macro' is safer
average_mode = "binary" if len(set(y_true.tolist())) == 2 else "macro"
pr, re, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=average_mode, zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print(f"Test Precision: {pr:.4f}, Recall: {re:.4f}, F1: {f1:.4f}")
print("Confusion Matrix:\n", cm)

Train: loss=0.0004, acc=1.0000
Val:   loss=1.1168, acc=0.8433
Test:  loss=0.7880, acc=0.8776
Test Precision: 0.8492, Recall: 0.9184, F1: 0.8824
Confusion Matrix:
 [[277  54]
 [ 27 304]]


In [16]:
import os

# Build the summary text
summary_lines = [
    f"Model: {cfg.model_name}",
    f"Best Val Acc: {best_val_acc:.4f}",
    f"Test Acc: {test_acc:.4f}",
    f"Test Precision: {pr:.4f}, Recall: {re:.4f}, F1: {f1:.4f} (avg={average_mode})",
    f"Confusion Matrix:\n{cm}",
    f"Logs: {log_csv}",
    f"Weights: {weights_path}",
    f"Predictions: {pred_json}",
]

summary_text = "\n".join(summary_lines)

# ✅ Print summary to notebook output
print("\n" + "="*80)
print("📊 EXPERIMENT SUMMARY")
print("="*80)
print(summary_text)
print("="*80)

# 📝 Also save it to file
summary_path = os.path.join(cfg.save_dir, "result_summary.txt")
os.makedirs(cfg.save_dir, exist_ok=True)
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary_text)

# ✅ Confirm save path
abs_path = os.path.abspath(summary_path)
print(f"\n✅ Summary also saved to:\n{abs_path}")


📊 EXPERIMENT SUMMARY
Model: roberta-large
Best Val Acc: 0.8433
Test Acc: 0.8776
Test Precision: 0.8492, Recall: 0.9184, F1: 0.8824 (avg=binary)
Confusion Matrix:
[[277  54]
 [ 27 304]]
Logs: ./runs/ladder_cls/training_logs.csv
Weights: ./runs/ladder_cls/weights.pt
Predictions: ./runs/ladder_cls/predictions.json

✅ Summary also saved to:
/home/jovyan/work/Code/TTPClassifier/runs/ladder_cls/result_summary.txt


In [ ]:
#testing the model

In [17]:
#=============================================================================
# SAVE FULL MODEL IN HUGGINGFACE FORMAT (for easy reloading later)
#=============================================================================

print("\n" + "="*70)
print("SAVING FULL MODEL")
print("="*70)

# Define save path
full_model_dir = os.path.join(cfg.save_dir, "full_model")

# Ensure best weights are loaded
if os.path.exists(weights_path):
    model.load_state_dict(torch.load(weights_path, map_location=device))
    print(f"✓ Loaded best weights from: {weights_path}")
else:
    print("⚠️  No weights file found, saving current model state")

# Save in HuggingFace format
model.save_pretrained(full_model_dir)
tokenizer.save_pretrained(full_model_dir)

print(f"\n✓ Full model saved to: {full_model_dir}")
print("\nFiles created:")
print(f"  - config.json           (model configuration)")
print(f"  - pytorch_model.bin     (model weights)")
print(f"  - model.safetensors     (model weights in safetensors format)")
print(f"  - tokenizer files       (vocab, merges, tokenizer_config)")

print("\n" + "-"*70)
print("To load this model later, use:")
print("-"*70)
print(f"from transformers import AutoModelForSequenceClassification, AutoTokenizer")
print(f"")
print(f"model = AutoModelForSequenceClassification.from_pretrained('{full_model_dir}')")
print(f"tokenizer = AutoTokenizer.from_pretrained('{full_model_dir}')")
print("="*70 + "\n")


SAVING FULL MODEL
✓ Loaded best weights from: ./runs/ladder_cls/weights.pt

✓ Full model saved to: ./runs/ladder_cls/full_model

Files created:
  - config.json           (model configuration)
  - pytorch_model.bin     (model weights)
  - model.safetensors     (model weights in safetensors format)
  - tokenizer files       (vocab, merges, tokenizer_config)

----------------------------------------------------------------------
To load this model later, use:
----------------------------------------------------------------------
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained('./runs/ladder_cls/full_model')
tokenizer = AutoTokenizer.from_pretrained('./runs/ladder_cls/full_model')



In [18]:
from transformers import pipeline

pipe = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    return_all_scores=True,      # keep per-class scores
    truncation=True
)

# Try a few sentences
my_sentences = [
    "Activating the device’s microphone and listening to live conversations",
    "I love sunny beaches and gelato.",
    "Copying files from the device to a Command & Control center"
]

raw = pipe(my_sentences)
raw

Device set to use cuda:0
/opt/conda/lib/python3.11/site-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


[[{'label': '0', 'score': 3.0080798751441762e-05},
  {'label': '1', 'score': 0.9999699592590332}],
 [{'label': '0', 'score': 0.9999362230300903},
  {'label': '1', 'score': 6.381928687915206e-05}],
 [{'label': '0', 'score': 3.604662197176367e-05},
  {'label': '1', 'score': 0.9999639987945557}]]

In [19]:
import os
import re

input_file = os.path.join(input_dir, "/home/jovyan/work/MyDataSets/MTest.txt")
output_file = "/home/jovyan/work/MyDataSets/MTest_sentences.txt"

NO_SPACE_BEFORE = {".", ",", ":", ";", "!", "?", "%", ")", "]", "}", "。", "、"}
NO_SPACE_AFTER = {"(", "[", "{"}

def detok_join(tokens):
    out = []
    for i, tok in enumerate(tokens):
        if i == 0:
            out.append(tok)
            continue
        if tok in NO_SPACE_BEFORE:
            out[-1] = out[-1] + tok
        elif out[-1][-1:] in NO_SPACE_AFTER:
            out[-1] = out[-1] + tok
        else:
            out.append(tok)
    return " ".join(out)

# 1) read tokens (ignore tags)
doc_tokens = []
with open(input_file, "r", encoding="utf-8") as infile:
    for line in infile:
        raw = line.rstrip("\n")  # remove only newline, keep tabs/spaces

        # completely blank (or whitespace-only) line → paragraph hint
        if raw.strip() == "":
            doc_tokens.append("\n")
            continue

        if "\t" in raw:
            # split into token + tag (and ignore the tag)
            token, tag_and_rest = raw.split("\t", 1)
            token = token.strip()

            # 🔴 token column is empty (like "\tO"): treat as paragraph break or skip
            if token == "":
                doc_tokens.append("\n")   # or `continue` if you prefer no break
                continue
        else:
            # no tab: fall back to whitespace split
            parts = raw.split()
            if not parts:
                continue

            # line like "O" alone (pure tag line) → skip
            if len(parts) == 1 and parts[0] == "O":
                continue

            token = parts[0]

        doc_tokens.append(token)


# 2) detokenize to raw text
raw_text = detok_join([t for t in doc_tokens if t != "\n"])

# 3) sentence split with regex:
#    split on [.?!] followed by space + uppercase or end of string.
sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z(\"\'])', raw_text)
# fallback if we somehow got only one mega sentence: also split on " . " sequences
if len(sentences) == 1:
    sentences = re.split(r'(?<=[.!?])\s+', raw_text)

# 4) cleanup & write
sentences = [s.strip() for s in sentences if s.strip()]

os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, "w", encoding="utf-8") as out:
    for s in sentences:
        out.write(s + "\n")

print(f"✅ Converted '{input_file}' → '{output_file}'")
print(f"📝 {len(sentences)} sentences written.\n")

print("📄 Sample sentences:")
for s in sentences[:10]:
    print("-", s[:200] + ("..." if len(s) > 200 else ""))


✅ Converted '/home/jovyan/work/MyDataSets/MTest.txt' → '/home/jovyan/work/MyDataSets/MTest_sentences.txt'
📝 279 sentences written.

📄 Sample sentences:
- Dropping Elephant APT Group Targets Turkish Defense Industry With New Campaign and Capabilities: LOLBAS, VLC Player, and Encrypted Shellcode - Arctic Wolf Search Experienced a Breach?
- Contact Us Blog EN EN-GB (United Kingdom) FR (Français) DE (Deutsch) Nederlands (Dutch) Suomi (Finnish) 日本語 (Japanese) Norsk (Norwegian) Svenska (Swedish) EN EN-GB (United Kingdom) FR (Français) DE (D...
- Aurora Platform Collect, enrich, and analyze security data at scale.
- Alpha AI Leverage the power of scale and AI expertise.
- Platform Integrations Ecosystem integrations and technology partnerships.
- Concierge Concierge Delivery Model Tailored security expertise and guided risk mitigation.
- Arctic Wolf Security Teams Security experts proactively protecting you 24×7.
- Go Inside Our SOC Meet the security experts working alongside you and your tea

In [20]:
from transformers import pipeline
import torch

# Initialize the text classification pipeline
pipe = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    return_all_scores=True,      # keep per-class scores for analysis
    truncation=True
)

# 🧠 Use the sentences from your sentence-splitting step
# (make sure you've run the previous cell where `sentences` was defined)
my_sentences = [s.strip() for s in sentences if s.strip()]

# Run classification
raw = pipe(my_sentences)

# 🖨️ Display each sentence with its top predicted label and score
print(f"✅ Classified {len(my_sentences)} sentences.\n")
for sent, preds in zip(my_sentences[:300], raw):   # show first 10 for readability
    top = max(preds, key=lambda x: x["score"])
    print(f"📝 Sentence: {sent}")
    print(f"🔹 Top label: {top['label']} (score={top['score']:.4f})")
    print(f"🔸 All scores: {[(p['label'], round(p['score'], 4)) for p in preds]}")
    print("-" * 80)

# Optional: show structured summary for all
summary = [
    {
        "sentence": sent,
        "top_label": max(preds, key=lambda x: x["score"])["label"],
        "top_score": max(preds, key=lambda x: x["score"])["score"],
        "all_scores": preds,
    }
    for sent, preds in zip(my_sentences, raw)
]

print("\n📊 Example structured output:")
for r in summary[:3]:
    print(r)


Device set to use cuda:0


✅ Classified 279 sentences.

📝 Sentence: Dropping Elephant APT Group Targets Turkish Defense Industry With New Campaign and Capabilities: LOLBAS, VLC Player, and Encrypted Shellcode - Arctic Wolf Search Experienced a Breach?
🔹 Top label: 1 (score=0.9983)
🔸 All scores: [('0', 0.0017), ('1', 0.9983)]
--------------------------------------------------------------------------------
📝 Sentence: Contact Us Blog EN EN-GB (United Kingdom) FR (Français) DE (Deutsch) Nederlands (Dutch) Suomi (Finnish) 日本語 (Japanese) Norsk (Norwegian) Svenska (Swedish) EN EN-GB (United Kingdom) FR (Français) DE (Deutsch) Nederlands (Dutch) Suomi (Finnish) 日本語 (Japanese) Norsk (Norwegian) Svenska (Swedish) Platform Platform How It Works Delivering security operations outcomes.
🔹 Top label: 0 (score=0.9999)
🔸 All scores: [('0', 0.9999), ('1', 0.0001)]
--------------------------------------------------------------------------------
📝 Sentence: Aurora Platform Collect, enrich, and analyze security data at scale.
🔹 To

In [21]:
import os
from collections import OrderedDict

assert "summary" in locals(), "⚠️ Run the classification cell first to create 'summary'."

# ---- Config ----
min_conf = 0.0          # set >0.0 to filter by confidence, e.g., 0.80
strip_leading_O = True  # remove a leading "O " artifact if present
out_dir = cfg.save_dir if "cfg" in globals() else "./runs/ladder_cls"
os.makedirs(out_dir, exist_ok=True)

# Helper to clean the sentence
def clean_sentence(s: str) -> str:
    s = s.strip()
    if strip_leading_O and s.startswith("O "):
        s = s[2:].lstrip()
    return s

# Collect by label, keep order, deduplicate
by_label = {}
for item in summary:
    lbl = str(item["top_label"])          # e.g., "0" or "1"
    score = float(item["top_score"])
    if score < min_conf:
        continue
    sent = clean_sentence(item["sentence"])
    if not sent:
        continue
    by_label.setdefault(lbl, OrderedDict())
    by_label[lbl].setdefault(sent, None)  # Ordered-set behavior

# Write one file per label
output_files = []
for lbl, od in sorted(by_label.items(), key=lambda x: x[0]):  # sort labels "0","1",...
    safe_lbl = lbl.replace("/", "_").replace(" ", "_")
    path = os.path.join(out_dir, f"sentences_{safe_lbl}.txt")
    with open(path, "w", encoding="utf-8") as f:
        for s in od.keys():
            f.write(s + "\n")
    output_files.append((lbl, path, len(od)))

# Report
print("✅ Files written:")
for lbl, path, n in output_files:
    print(f"  Label '{lbl}': {n} sentences → {os.path.abspath(path)}")

# Quick preview
for lbl, path, n in output_files:
    print(f"\n📄 Sample from label {lbl} ({n} sentences):")
    try:
        with open(path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                print("-", line.strip()[:200] + ("..." if len(line.strip()) > 200 else ""))
                if i >= 4:
                    break
    except FileNotFoundError:
        print("  (no file)")


✅ Files written:
  Label '0': 195 sentences → /home/jovyan/work/Code/TTPClassifier/runs/ladder_cls/sentences_0.txt
  Label '1': 84 sentences → /home/jovyan/work/Code/TTPClassifier/runs/ladder_cls/sentences_1.txt

📄 Sample from label 0 (195 sentences):
- Contact Us Blog EN EN-GB (United Kingdom) FR (Français) DE (Deutsch) Nederlands (Dutch) Suomi (Finnish) 日本語 (Japanese) Norsk (Norwegian) Svenska (Swedish) EN EN-GB (United Kingdom) FR (Français) DE (D...
- Alpha AI Leverage the power of scale and AI expertise.
- Platform Integrations Ecosystem integrations and technology partnerships.
- Concierge Concierge Delivery Model Tailored security expertise and guided risk mitigation.
- Arctic Wolf Security Teams Security experts proactively protecting you 24×7.

📄 Sample from label 1 (84 sentences):
- Dropping Elephant APT Group Targets Turkish Defense Industry With New Campaign and Capabilities: LOLBAS, VLC Player, and Encrypted Shellcode - Arctic Wolf Search Experienced a Breach?
- Aurora Pla